In [ ]:
# This script is developped by Omar Sallam 
# osallam@tamu.edu ;  osallam@anl.gov ; omarkhaledsallam@gmail.com 

In [ ]:

from __future__ import annotations
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point
from pygeohydro import WBD
import py3dep
import pygeoogc.utils as ogc_utils
import pynhd
from pynhd import NLDI, WaterData
import rioxarray
from py3dep import get_dem ,deg2mpm
import os
import numpy as np
import pandas as pd
import xarray as xr
from glob import glob
from pynhd.network_tools import prepare_nhdplus
from shapely.geometry import Point, LineString
import matplotlib.pyplot as plt
from pynhd import NHD
import rasterio
from rasterio import features
from datetime import datetime, timedelta
import sys  
from shapely.geometry import Polygon
import contextily as cx
import random
import requests
import simplekml
import matplotlib.ticker as ticker


In [ ]:

truth_vals = np.load(Path("GT_sample_data_folder") / "truth_traj120.npy", mmap_mode="r")

pred_vals = np.load(Path("GT_sample_data_folder") / "predictions_traj120.npy", mmap_mode="r")


In [ ]:

print(truth_vals.shape)


In [ ]:
plt.rcParams.update({
    'axes.labelsize': 14,
    'font.size': 14,
    'mathtext.fontset': 'cm',
    'mathtext.default': 'it',
    'font.family': 'serif',
    'font.weight': 'normal'
})

In [ ]:

# Inputs 
crs_proj = 5070
resolution = 30
# --- Simulation setup ---
filling_hours = 0  # hours
start_date = '2013-04-10'
end_date = '2013-04-23'
watershed_id = "07120004" # Desplaines # Example of HUC8
huc_digit_number  = len(watershed_id)

main_river_gauges_to_plot = [0,7,14,19,20,25]
width_90_gauges = [0, 19, 20, 25]

parent_dir = 'GT_sample_data_folder/'
dem_path  = parent_dir + 'dem_5070.tif'
predictions_path = parent_dir + 'predictions_traj120.npy'
predictions_vanilla_path = parent_dir + 'predictions_vanilla_traj120.npy'
synxflow_dir = parent_dir + 'combined_files' # kept for downstream flood-extent map cells
output_dir = parent_dir + 'output'
figdir = os.path.join(output_dir,"Figures_CLDNet/")
kmldir = os.path.join(output_dir,"KML/")
os.makedirs(figdir, exist_ok=True)
os.makedirs(kmldir, exist_ok=True)

comparison_start_datetime = pd.Timestamp('2013-04-17 08:00:00')
comparison_end_datetime = pd.Timestamp('2013-04-21 07:00:00')
usgs_timezone = 'America/Chicago'
usgs_resample_rule = '1h'

### USGS Folders ###
usgs_folder_path_h = parent_dir + rf"USGS_2013-04-15_2013-04-23/h/"
usgs_folder_path_q = parent_dir + rf"USGS_2013-04-15_2013-04-23/q/"


In [ ]:

real_photo_locations = {'mac': (42.046400 , -87.885580),
                        # 'aldi': (41.833045, -88.104890),
                        'electric_park': (41.606825, -88.209490),
                        'subway': (41.929179, -87.846377),
                        'lisle_PD': (41.793778, -88.077573),
                        # 'big_bend_lake': (42.050956, -87.877275),
                        'gurnee_grad_school': (42.373595, -87.918682),
                        # 'river_grove': (41.9200, -88.0200)
                        }



In [ ]:

dem = rioxarray.open_rasterio(dem_path)
pred_vals = np.load(predictions_path, mmap_mode="r")
ldnet_vals = np.load(predictions_vanilla_path, mmap_mode="r")
truth_path = os.path.join(parent_dir, "truth_traj120.npy")
truth_vals = np.load(truth_path, mmap_mode="r")

cldnet_n_frames = pred_vals.shape[0]
ldnet_n_frames = ldnet_vals.shape[0]
synxflow_n_frames = truth_vals.shape[0]
cldnet_frame_times = pd.date_range(start=comparison_start_datetime, periods=cldnet_n_frames, freq="1h")
ldnet_frame_times = pd.date_range(start=comparison_start_datetime, periods=ldnet_n_frames, freq="1h")
synxflow_frame_times = pd.date_range(start=comparison_start_datetime, periods=synxflow_n_frames, freq="1h")

print(f"Loaded CLDNet prediction trajectory with shape {pred_vals.shape}.")
print(f"Loaded vanilla LDNet prediction trajectory with shape {ldnet_vals.shape}.")
print(f"Loaded SynxFlow truth trajectory with shape {truth_vals.shape}.")
print(
    f"Generated {len(cldnet_frame_times)} CLDNet hourly timestamps from "
    f"{cldnet_frame_times[0]} to {cldnet_frame_times[-1]}."
)
print(
    f"Generated {len(ldnet_frame_times)} LDNet hourly timestamps from "
    f"{ldnet_frame_times[0]} to {ldnet_frame_times[-1]}."
)
print(
    f"Generated {len(synxflow_frame_times)} SynxFlow hourly timestamps from "
    f"{synxflow_frame_times[0]} to {synxflow_frame_times[-1]}."
)
if comparison_end_datetime != cldnet_frame_times[-1]:
    print(
        f"Requested end time {comparison_end_datetime} differs from the last CLDNet simulation timestamp "
        f"{cldnet_frame_times[-1]} because predictions_traj120.npy contains {cldnet_n_frames} hourly steps."
    )
if comparison_end_datetime != ldnet_frame_times[-1]:
    print(
        f"Requested end time {comparison_end_datetime} differs from the last LDNet simulation timestamp "
        f"{ldnet_frame_times[-1]} because predictions_vanilla_traj120.npy contains {ldnet_n_frames} hourly steps."
    )
if comparison_end_datetime != synxflow_frame_times[-1]:
    print(
        f"Requested end time {comparison_end_datetime} differs from the last SynxFlow simulation timestamp "
        f"{synxflow_frame_times[-1]} because truth_traj120.npy contains {synxflow_n_frames} hourly steps."
    )


In [ ]:

wbd = WBD(rf"huc{huc_digit_number}")
huc = wbd.byids(rf"huc{huc_digit_number}", [watershed_id])  # 

nldi = NLDI()
nhd = NHD("flowline_mr")
flowlines = nhd.bygeom(huc.geometry.iloc[0])
flowlines = prepare_nhdplus(
    flowlines,
    min_network_size=0,
    min_path_length=0,
    remove_isolated=True,
)
flowlines = flowlines.to_crs(5070)

In [ ]:

def cross_at_point(flowline, point, half_width=1000):
    """Create a perpendicular cross-section at the given point along the given flowline."""
    d = flowline.project(point)
    nearest = flowline.interpolate(d)
    tangent = flowline.interpolate(d + 1)
    dx, dy = tangent.x - nearest.x, tangent.y - nearest.y
    L = np.hypot(dx, dy)
    nx, ny = -dy / L, dx / L
    p1 = (nearest.x + nx * half_width, nearest.y + ny * half_width)
    p2 = (nearest.x - nx * half_width, nearest.y - ny * half_width)
    return LineString([p1, p2])

def cross_sections_from_points(flowlines, points_gdf, half_width=1000):
    """Generate perpendicular cross-sections for all points in a GeoDataFrame."""
    cs_geoms = []
    nearest_ids = []
    for i, pt in enumerate(points_gdf.geometry):
        flowlines["dist_to_point"] = flowlines.distance(pt)
        nearest_idx = flowlines["dist_to_point"].idxmin()
        nearest_flowline = flowlines.loc[nearest_idx].geometry

        cs_line = cross_at_point(nearest_flowline, pt, half_width)
        cs_geoms.append(cs_line)
        nearest_ids.append(nearest_idx)

    cs = gpd.GeoDataFrame(
        {"nearest_comid": nearest_ids, "geometry": cs_geoms},
        crs=flowlines.crs,
    )
    return cs

def count_points(geom):
    if geom.geom_type == "LineString":
        return len(geom.coords)
    elif geom.geom_type == "MultiLineString":
        return sum(len(g.coords) for g in geom.geoms)
    else:
        return 0


def get_line_indices(line, dem_path):
    """Return all (row, col) indices that a line crosses in the raster."""
    with rasterio.open(dem_path) as src:
        mask = features.rasterize(
            [(line, 1)],
            out_shape=(src.height, src.width),
            transform=src.transform,
            all_touched=True,
            fill=0,
            dtype="uint8",
        )
        rows, cols = np.where(mask == 1)
        return rows.tolist(), cols.tolist()


def extract_crosssection_indices_grouped(cs_gdf, dem, max_number_points):
    """
    Return one row per COMID with lists of y_indices and x_indices
    for all raster pixels each cross-section intersects.
    """
    dem_path = dem.rio._manager.acquire().name
    records = []

    for comid, geom in cs_gdf.geometry.items():
        lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]

        all_rows, all_cols = [], []
        for line in lines:
            rows, cols = get_line_indices(line, dem_path)
            rows = rows[0:max_number_points]
            cols = cols[0:max_number_points]
            all_rows.extend(rows)
            all_cols.extend(cols)

        records.append({
            "comid": comid,
            "y_indices": all_rows,
            "x_indices": all_cols
        })

    return pd.DataFrame(records)


def _to_local_naive_datetime(values, timezone=usgs_timezone):
    dt = pd.to_datetime(values, errors="coerce")
    if hasattr(dt, "dt") and dt.dt.tz is not None:
        return dt.dt.tz_convert(timezone).dt.tz_localize(None)
    return dt


def _fetch_usgs_iv(site_no, parameter_cd, value_column, start_dt, end_dt):
    url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": "json",
        "sites": str(site_no),
        "parameterCd": parameter_cd,
        "startDT": start_dt.strftime("%Y-%m-%dT%H:%M"),
        "endDT": end_dt.strftime("%Y-%m-%dT%H:%M"),
        "siteStatus": "all",
    }

    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
    except Exception as exc:
        print(f"USGS IV request failed for site {site_no}, parameter {parameter_cd}: {exc}")
        return pd.DataFrame(columns=["datetime", value_column])

    records = []
    for series in payload.get("value", {}).get("timeSeries", []):
        for block in series.get("values", []):
            for entry in block.get("value", []):
                raw_value = entry.get("value")
                if raw_value in (None, "", "Eqp", "Ice"):
                    continue
                try:
                    numeric_value = float(raw_value)
                except ValueError:
                    continue
                records.append({"datetime": entry.get("dateTime"), value_column: numeric_value})

    if not records:
        return pd.DataFrame(columns=["datetime", value_column])

    df = pd.DataFrame(records)
    df["datetime"] = _to_local_naive_datetime(df["datetime"])
    return df.dropna(subset=["datetime"]).sort_values("datetime")


def _finalize_usgs_timeseries(df, value_column, start_dt=comparison_start_datetime, end_dt=comparison_end_datetime):
    if df.empty:
        return df

    df = df.copy()
    df["datetime"] = _to_local_naive_datetime(df["datetime"])
    df[value_column] = pd.to_numeric(df[value_column], errors="coerce")
    df = df.dropna(subset=["datetime", value_column]).sort_values("datetime")
    df = df[(df["datetime"] >= start_dt) & (df["datetime"] <= end_dt)]
    if df.empty:
        return df

    value_frame = df[["datetime", value_column]].drop_duplicates(subset="datetime", keep="last")
    value_frame = value_frame.set_index("datetime").resample(usgs_resample_rule).nearest()
    value_frame = value_frame.loc[start_dt:end_dt]

    meta_cols = [c for c in df.columns if c not in {"datetime", value_column}]
    if meta_cols:
        meta_frame = df[["datetime"] + meta_cols].drop_duplicates(subset="datetime", keep="last")
        meta_frame = meta_frame.set_index("datetime").resample(usgs_resample_rule).ffill().bfill()
        meta_frame = meta_frame.loc[start_dt:end_dt]
        value_frame = value_frame.join(meta_frame, how="left")

    return value_frame.reset_index()


def get_usgs_h_wse(usgs_folder_path, usgs_df_paths, gage_index):
    csv_path = os.path.join(usgs_folder_path, usgs_df_paths[gage_index])
    site_no = usgs_df_paths[gage_index].split("_")[1]

    local_df = pd.read_csv(csv_path)
    remote_df = _fetch_usgs_iv(site_no, "00065", "Gage Height (ft)", comparison_start_datetime, comparison_end_datetime)

    if not remote_df.empty:
        meta = local_df.iloc[0].to_dict()
        usgs_df = remote_df.copy()
        for col, value in meta.items():
            if col not in usgs_df.columns:
                usgs_df[col] = value
    else:
        usgs_df = local_df

    usgs_df = _finalize_usgs_timeseries(usgs_df, "Gage Height (ft)")
    if usgs_df.empty:
        return usgs_df

    if gage_index in {14, 20}:
        alt_datum = (usgs_df['alt_va'].iloc[0] + 10) * 0.3048
    else:
        alt_datum = usgs_df['alt_va'].iloc[0] * 0.3048

    usgs_df["stage"] = usgs_df['Gage Height (ft)'] * 0.3048
    usgs_df["wse"] = usgs_df["stage"] + alt_datum
    return usgs_df


def get_usgs_q(usgs_folder_path, usgs_df_paths, gage_index):
    csv_path = os.path.join(usgs_folder_path, usgs_df_paths[gage_index])
    site_no = usgs_df_paths[gage_index].split("_")[1]

    local_df = pd.read_csv(csv_path)
    remote_df = _fetch_usgs_iv(site_no, "00060", "Discharge (cfs)", comparison_start_datetime, comparison_end_datetime)

    if not remote_df.empty:
        meta = local_df.iloc[0].to_dict()
        usgs_df = remote_df.copy()
        usgs_df["Discharge (cms)"] = usgs_df.pop("Discharge (cfs)") * 0.028316846592
        for col, value in meta.items():
            if col not in usgs_df.columns:
                usgs_df[col] = value
    else:
        usgs_df = local_df

    usgs_df = _finalize_usgs_timeseries(usgs_df, "Discharge (cms)")
    if usgs_df.empty:
        return usgs_df

    usgs_df["Q"] = usgs_df['Discharge (cms)']
    return usgs_df


In [ ]:

# Configuration/Constants
WIDTH_VALUES = [60, 90]
dem_grid = np.squeeze(dem.values)

# If these NetCDFs were opened earlier in the notebook, close them before overwriting.
for dataset_dict_name in ["cldnet_datasets", "ldnet_datasets", "synxflow_datasets"]:
    existing_dict = globals().get(dataset_dict_name)
    if isinstance(existing_dict, dict):
        for existing_ds in existing_dict.values():
            try:
                existing_ds.close()
            except Exception:
                pass
        globals()[dataset_dict_name] = {}

for width in WIDTH_VALUES:
    # 1. SETUP & GEOPROCESSING
    # ---------------------------------------------------------
    max_number_points = int(width / resolution)

    # Load gauge locations
    points_path = parent_dir + 'gages.geojson'
    points_gdf = gpd.read_file(points_path)

    # Clean geometries: Convert MultiPoint to single Point
    points_gdf["geometry"] = points_gdf["geometry"].apply(
        lambda g: g.geoms[0] if g.geom_type == "MultiPoint" else g
    )

    # Standardize Coordinate Reference Systems (CRS)
    points_gdf = points_gdf.set_crs(4326, allow_override=True)
    flowlines = flowlines.to_crs(5070)
    points_gdf = points_gdf.to_crs(flowlines.crs)

    # Generate cross-section lines and map to DEM indices
    cs = cross_sections_from_points(flowlines, points_gdf, half_width=width / 2)
    cs["n_points"] = cs.geometry.apply(count_points)
    cs_indices = extract_crosssection_indices_grouped(cs, dem, max_number_points)

    # Visualization of the domain and gauges
    fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
    flowlines.plot(ax=ax, color="lightgray", linewidth=0.5)
    points_gdf.plot(ax=ax, color="green", markersize=50, label="Gauges")
    cs.plot(ax=ax, color="red", linewidth=0.5, label="Cross Sections")
    ax.set_aspect("equal")
    plt.legend()
    plt.show()

    # 2. TIME METADATA PREPARATION
    # ---------------------------------------------------------
    cldnet_time_step = (
        cldnet_frame_times[1] - cldnet_frame_times[0]
        if len(cldnet_frame_times) > 1
        else pd.Timedelta(0)
    )
    ldnet_time_step = (
        ldnet_frame_times[1] - ldnet_frame_times[0]
        if len(ldnet_frame_times) > 1
        else pd.Timedelta(0)
    )
    synxflow_time_step = (
        synxflow_frame_times[1] - synxflow_frame_times[0]
        if len(synxflow_frame_times) > 1
        else pd.Timedelta(0)
    )
    print(f"Detected CLDNet time step: {cldnet_time_step}")
    print(f"Detected LDNet time step: {ldnet_time_step}")
    print(f"Detected SynxFlow time step: {synxflow_time_step}")

    # 3. CORE DATA EXTRACTION FROM predictions_traj120.npy
    # ---------------------------------------------------------
    cldnet_n_frames = pred_vals.shape[0]
    n_gauges = len(cs_indices.y_indices)
    max_len = max(len(y_idx) for y_idx in cs_indices.y_indices)

    cldnet_h_array = np.full((cldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    cldnet_hU_array = np.full((cldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    cldnet_dem_array = np.full((cldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)

    print(f"Processing {cldnet_n_frames} CLDNet prediction frames for {n_gauges} gauges...")

    for fidx in range(cldnet_n_frames):
        frame_h = pred_vals[fidx, :, :, 0]
        frame_u = pred_vals[fidx, :, :, 1]
        frame_v = pred_vals[fidx, :, :, 2]

        for g_idx in range(n_gauges):
            y_pix = cs_indices.y_indices[g_idx]
            x_pix = cs_indices.x_indices[g_idx]
            num_pix = len(y_pix)

            h_vals = frame_h[y_pix, x_pix]
            u_vals = frame_u[y_pix, x_pix]
            v_vals = frame_v[y_pix, x_pix]
            dem_vals = dem_grid[y_pix, x_pix]

            hU_vals = h_vals * np.sqrt(u_vals**2 + v_vals**2)

            cldnet_h_array[fidx, g_idx, :num_pix] = h_vals
            cldnet_hU_array[fidx, g_idx, :num_pix] = hU_vals
            cldnet_dem_array[fidx, g_idx, :num_pix] = dem_vals

        if fidx % 10 == 0:
            print(f"  Processed CLDNet prediction frame {fidx}/{cldnet_n_frames}")

    cldnet_ds = xr.Dataset(
        {
            "h": (("frame", "gauge", "point"), cldnet_h_array),
            "hU": (("frame", "gauge", "point"), cldnet_hU_array),
            "dem": (("frame", "gauge", "point"), cldnet_dem_array),
        },
        coords={
            "frame": np.arange(cldnet_n_frames),
            "time": ("frame", cldnet_frame_times),
            "gauge": np.arange(n_gauges),
            "point": np.arange(max_len),
        },
        attrs={"time_step": str(cldnet_time_step), "resolution": resolution, "source": "predictions_traj120.npy"}
    )

    cldnet_wse = cldnet_ds.h.max(dim="point", skipna=True) + cldnet_ds.dem.mean(dim="point", skipna=True)
    cldnet_q = cldnet_ds.hU.fillna(0).sum(dim="point") * resolution

    # 4. CORE DATA EXTRACTION FROM predictions_vanilla_traj120.npy
    # ---------------------------------------------------------
    ldnet_n_frames = ldnet_vals.shape[0]
    ldnet_h_array = np.full((ldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    ldnet_hU_array = np.full((ldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    ldnet_dem_array = np.full((ldnet_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)

    print(f"Processing {ldnet_n_frames} LDNet prediction frames for {n_gauges} gauges...")

    for fidx in range(ldnet_n_frames):
        frame_h = ldnet_vals[fidx, :, :, 0]
        frame_u = ldnet_vals[fidx, :, :, 1]
        frame_v = ldnet_vals[fidx, :, :, 2]

        for g_idx in range(n_gauges):
            y_pix = cs_indices.y_indices[g_idx]
            x_pix = cs_indices.x_indices[g_idx]
            num_pix = len(y_pix)

            h_vals = frame_h[y_pix, x_pix]
            u_vals = frame_u[y_pix, x_pix]
            v_vals = frame_v[y_pix, x_pix]
            dem_vals = dem_grid[y_pix, x_pix]

            hU_vals = h_vals * np.sqrt(u_vals**2 + v_vals**2)

            ldnet_h_array[fidx, g_idx, :num_pix] = h_vals
            ldnet_hU_array[fidx, g_idx, :num_pix] = hU_vals
            ldnet_dem_array[fidx, g_idx, :num_pix] = dem_vals

        if fidx % 10 == 0:
            print(f"  Processed LDNet prediction frame {fidx}/{ldnet_n_frames}")

    ldnet_ds = xr.Dataset(
        {
            "h": (("frame", "gauge", "point"), ldnet_h_array),
            "hU": (("frame", "gauge", "point"), ldnet_hU_array),
            "dem": (("frame", "gauge", "point"), ldnet_dem_array),
        },
        coords={
            "frame": np.arange(ldnet_n_frames),
            "time": ("frame", ldnet_frame_times),
            "gauge": np.arange(n_gauges),
            "point": np.arange(max_len),
        },
        attrs={"time_step": str(ldnet_time_step), "resolution": resolution, "source": "predictions_vanilla_traj120.npy"}
    )

    ldnet_wse = ldnet_ds.h.max(dim="point", skipna=True) + ldnet_ds.dem.mean(dim="point", skipna=True)
    ldnet_q = ldnet_ds.hU.fillna(0).sum(dim="point") * resolution

    # 5. CORE DATA EXTRACTION FROM truth_traj120.npy
    # ---------------------------------------------------------
    synxflow_n_frames = truth_vals.shape[0]
    synxflow_h_array = np.full((synxflow_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    synxflow_hU_array = np.full((synxflow_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)
    synxflow_dem_array = np.full((synxflow_n_frames, n_gauges, max_len), np.nan, dtype=np.float32)

    print(f"Processing {synxflow_n_frames} SynxFlow truth frames for {n_gauges} gauges...")

    for fidx in range(synxflow_n_frames):
        frame_h = truth_vals[fidx, :, :, 0]
        frame_u = truth_vals[fidx, :, :, 1]
        frame_v = truth_vals[fidx, :, :, 2]

        for g_idx in range(n_gauges):
            y_pix = cs_indices.y_indices[g_idx]
            x_pix = cs_indices.x_indices[g_idx]
            num_pix = len(y_pix)

            h_vals = frame_h[y_pix, x_pix]
            u_vals = frame_u[y_pix, x_pix]
            v_vals = frame_v[y_pix, x_pix]
            dem_vals = dem_grid[y_pix, x_pix]

            hU_vals = h_vals * np.sqrt(u_vals**2 + v_vals**2)

            synxflow_h_array[fidx, g_idx, :num_pix] = h_vals
            synxflow_hU_array[fidx, g_idx, :num_pix] = hU_vals
            synxflow_dem_array[fidx, g_idx, :num_pix] = dem_vals

        if fidx % 10 == 0:
            print(f"  Processed SynxFlow truth frame {fidx}/{synxflow_n_frames}")

    synxflow_ds = xr.Dataset(
        {
            "h": (("frame", "gauge", "point"), synxflow_h_array),
            "hU": (("frame", "gauge", "point"), synxflow_hU_array),
            "dem": (("frame", "gauge", "point"), synxflow_dem_array),
        },
        coords={
            "frame": np.arange(synxflow_n_frames),
            "time": ("frame", synxflow_frame_times),
            "gauge": np.arange(n_gauges),
            "point": np.arange(max_len),
        },
        attrs={"time_step": str(synxflow_time_step), "resolution": resolution, "source": "truth_traj120.npy"}
    )

    synxflow_wse = synxflow_ds.h.max(dim="point", skipna=True) + synxflow_ds.dem.mean(dim="point", skipna=True)
    synxflow_q = synxflow_ds.hU.fillna(0).sum(dim="point") * resolution

    # 6. EXPORT & FILE MANAGEMENT
    # ---------------------------------------------------------
    os.makedirs(output_dir, exist_ok=True)

    cldnet_save_path = os.path.join(output_dir, f"cross_section_metrics_cldnet_width_{width}.nc")
    ldnet_save_path = os.path.join(output_dir, f"cross_section_metrics_ldnet_width_{width}.nc")
    synxflow_save_path = os.path.join(output_dir, f"cross_section_metrics_synxflow_width_{width}.nc")

    # xr.Dataset({"wse": cldnet_wse, "Q": cldnet_q}).to_netcdf(cldnet_save_path, mode="w")
    # xr.Dataset({"wse": ldnet_wse, "Q": ldnet_q}).to_netcdf(ldnet_save_path, mode="w")
    # xr.Dataset({"wse": synxflow_wse, "Q": synxflow_q}).to_netcdf(synxflow_save_path, mode="w")

    # cldnet_ds.close()
    # ldnet_ds.close()
    # synxflow_ds.close()

    # print(f"Saved CLDNet results to: {cldnet_save_path}")
    # print(f"Saved LDNet results to: {ldnet_save_path}")
    # print(f"Saved SynxFlow results to: {synxflow_save_path}")


In [ ]:

# print(start_date_str)
# print(usgs_folder_path_h)
usgs_df_paths_h = os.listdir(usgs_folder_path_h)
usgs_df_paths_q = os.listdir(usgs_folder_path_q)
# print the csv files in the usgs dir in ascending order
usgs_df_paths_h = sorted(
    usgs_df_paths_h,
    key=lambda x: int(x.split("_")[0])   # sort by the number before "_"
)
usgs_df_paths_q = sorted(
    usgs_df_paths_q,
    key=lambda x: int(x.split("_")[0])   # sort by the number before "_"
)


In [ ]:
main_river_gauges_to_plot

In [ ]:

for dataset_dict_name in ["cldnet_datasets", "ldnet_datasets", "synxflow_datasets"]:
    existing_dict = globals().get(dataset_dict_name)
    if isinstance(existing_dict, dict):
        for existing_ds in existing_dict.values():
            try:
                existing_ds.close()
            except Exception:
                pass

cldnet_datasets = {
    60: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_cldnet_width_60.nc")),
    90: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_cldnet_width_90.nc")),
}
ldnet_datasets = {
    60: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_ldnet_width_60.nc")),
    90: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_ldnet_width_90.nc")),
}
synxflow_datasets = {
    60: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_synxflow_width_60.nc")),
    90: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_synxflow_width_90.nc")),
}

def _get_wse_series(ds, gage_idx, target_time):
    time_values = pd.to_datetime(ds.time.values)
    start_index = np.searchsorted(time_values, target_time)
    sim_wse = ds["wse"].isel(gauge=gage_idx, frame=slice(start_index, None))
    return pd.DataFrame({
        "datetime": pd.to_datetime(sim_wse["time"].values),
        "sim_wse": sim_wse.values,
    })


def _mean_shift_sim_wse(sim_df, synxflow_mean, usgs_mean):
    if sim_df.empty or np.isnan(synxflow_mean) or np.isnan(usgs_mean):
        return sim_df
    shifted_df = sim_df.copy()
    shifted_df["sim_wse"] = shifted_df["sim_wse"] - synxflow_mean + usgs_mean
    return shifted_df


def _align_wse_series(sim_df, usgs_df):
    merged = pd.merge(
        sim_df[["datetime", "sim_wse"]],
        usgs_df[["datetime", "wse"]],
        on="datetime",
        how="inner",
    ).dropna()
    return merged.rename(columns={"wse": "obs_wse"})


def _compute_nse(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    denom = np.sum((obs - np.mean(obs)) ** 2)
    if len(obs) < 2 or np.isclose(denom, 0.0):
        return np.nan
    return 1.0 - np.sum((sim - obs) ** 2) / denom


def _compute_kge(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    if len(obs) < 2 or np.isclose(np.std(obs, ddof=0), 0.0) or np.isclose(np.mean(obs), 0.0):
        return np.nan
    r = np.corrcoef(sim, obs)[0, 1]
    alpha = np.std(sim, ddof=0) / np.std(obs, ddof=0)
    beta = np.mean(sim) / np.mean(obs)
    if np.isnan(r):
        return np.nan
    return 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)


def _compute_peak_wse_relative_error(obs_df, sim_df):
    if obs_df.empty or sim_df.empty:
        return np.nan
    obs_peak = obs_df["wse"].max()
    obs_min = obs_df["wse"].min()
    sim_peak = sim_df["sim_wse"].max()
    peak_range = obs_peak - obs_min
    if np.isclose(peak_range, 0.0):
        return np.nan
    return abs((sim_peak - obs_peak) / peak_range)


def _compute_time_to_peak_error_hours(obs_df, sim_df):
    if obs_df.empty or sim_df.empty:
        return np.nan
    obs_peak_time = pd.to_datetime(obs_df.loc[obs_df["wse"].idxmax(), "datetime"])
    sim_peak_time = pd.to_datetime(sim_df.loc[sim_df["sim_wse"].idxmax(), "datetime"])
    return abs((sim_peak_time - obs_peak_time).total_seconds() / 3600.0)


def compute_discharge_metrics_for_gauge(gage_idx):
    width = 90 if gage_idx in width_90_gauges else 60
    cldnet_ds = cldnet_datasets[width]
    ldnet_ds = ldnet_datasets[width]
    synxflow_ds = synxflow_datasets[width]
    usgs_wse_df = get_usgs_h_wse(usgs_folder_path_h, usgs_df_paths_h, gage_idx)

    cldnet_sim_df = _get_wse_series(cldnet_ds, gage_idx, comparison_start_datetime)
    ldnet_sim_df = _get_wse_series(ldnet_ds, gage_idx, comparison_start_datetime)
    synxflow_sim_df = _get_wse_series(synxflow_ds, gage_idx, comparison_start_datetime)

    synxflow_mean = synxflow_sim_df["sim_wse"].mean()
    cldnet_mean = cldnet_sim_df["sim_wse"].mean()
    ldnet_mean = ldnet_sim_df["sim_wse"].mean()
    usgs_gauge_mean = usgs_wse_df["wse"].mean() if not usgs_wse_df.empty else np.nan
    cldnet_sim_df = _mean_shift_sim_wse(cldnet_sim_df, cldnet_mean, usgs_gauge_mean)
    ldnet_sim_df = _mean_shift_sim_wse(ldnet_sim_df, ldnet_mean, usgs_gauge_mean)
    synxflow_sim_df = _mean_shift_sim_wse(synxflow_sim_df, synxflow_mean, usgs_gauge_mean)

    cldnet_aligned = _align_wse_series(cldnet_sim_df, usgs_wse_df)
    ldnet_aligned = _align_wse_series(ldnet_sim_df, usgs_wse_df)
    synxflow_aligned = _align_wse_series(synxflow_sim_df, usgs_wse_df)


    site_no = (
        usgs_wse_df["site_no"].iloc[0]
        if not usgs_wse_df.empty and "site_no" in usgs_wse_df.columns
        else usgs_df_paths_h[gage_idx].split("_")[1]
    )

    records = []
    for model_name, sim_df, aligned_df in [
        ("CLDNet", cldnet_sim_df, cldnet_aligned),
        ("LDNet", ldnet_sim_df, ldnet_aligned),
        ("SynxFlow", synxflow_sim_df, synxflow_aligned),
    ]:
        records.append({
            "gage_index": gage_idx,
            "site_no": site_no,
            "model": model_name,
            "n_points_aligned": len(aligned_df),
            "nse": _compute_nse(aligned_df["obs_wse"], aligned_df["sim_wse"]),
            "kge": _compute_kge(aligned_df["obs_wse"], aligned_df["sim_wse"]),
            "peak_discharge_relative_error": _compute_peak_wse_relative_error(usgs_wse_df, sim_df),
            "time_to_peak_error_hours": _compute_time_to_peak_error_hours(usgs_wse_df, sim_df),
        })
    return records


metrics_records = []
for gage_idx in main_river_gauges_to_plot:
    metrics_records.extend(compute_discharge_metrics_for_gauge(gage_idx))

discharge_metrics_df = pd.DataFrame(metrics_records)
metrics_csv_path = os.path.join(output_dir, "gauge_discharge_metrics.csv")
discharge_metrics_df.to_csv(metrics_csv_path, index=False)

average_discharge_metrics_df = (
    discharge_metrics_df
    .groupby("model", as_index=False)[
        ["nse", "kge", "peak_discharge_relative_error", "time_to_peak_error_hours"]
    ]
    .mean(numeric_only=True)
)
average_metrics_csv_path = os.path.join(output_dir, "gauge_discharge_metrics_average.csv")
average_discharge_metrics_df.to_csv(average_metrics_csv_path, index=False)

print(discharge_metrics_df)
print(f"Saved discharge metrics to: {metrics_csv_path}")
print(average_discharge_metrics_df)
print(f"Saved average discharge metrics to: {average_metrics_csv_path}")


In [ ]:

for dataset_dict_name in ["cldnet_datasets", "ldnet_datasets", "synxflow_datasets"]:
    existing_dict = globals().get(dataset_dict_name)
    if isinstance(existing_dict, dict):
        for existing_ds in existing_dict.values():
            try:
                existing_ds.close()
            except Exception:
                pass


# --- Configuration ---
cldnet_datasets = {
    60: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_cldnet_width_60.nc")),
    90: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_cldnet_width_90.nc"))
}
synxflow_datasets = {
    60: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_synxflow_width_60.nc")),
    90: xr.open_dataset(os.path.join(output_dir, "cross_section_metrics_synxflow_width_90.nc"))
}

def plot_gauge_comparison(metric_name, y_label, filename, shift_mean=False):
    """
    Helper function to generate a row of subplots comparing CLDNet, SynxFlow, and USGS data.
    """
    n_gauges = len(main_river_gauges_to_plot)
    fig, axes = plt.subplots(1, n_gauges, figsize=(28, 3), dpi=300, sharex=True)

    if n_gauges == 1:
        axes = [axes]

    for i, gage_idx in enumerate(main_river_gauges_to_plot):
        ax = axes[i]
        width = 90 if gage_idx in width_90_gauges else 60
        cldnet_ds = cldnet_datasets[width]
        synxflow_ds = synxflow_datasets[width]

        cldnet_time_values = pd.to_datetime(cldnet_ds.time.values)
        synxflow_time_values = pd.to_datetime(synxflow_ds.time.values)
        cldnet_start_index = np.searchsorted(cldnet_time_values, comparison_start_datetime)
        synxflow_start_index = np.searchsorted(synxflow_time_values, comparison_start_datetime)

        cldnet_sim_data = cldnet_ds[metric_name].isel(gauge=gage_idx, frame=slice(cldnet_start_index, None))
        synxflow_sim_data = synxflow_ds[metric_name].isel(gauge=gage_idx, frame=slice(synxflow_start_index, None))

        if metric_name == "wse":
            usgs_df = get_usgs_h_wse(usgs_folder_path_h, usgs_df_paths_h, gage_idx)
            usgs_col = "wse"
        else:
            usgs_df = get_usgs_q(usgs_folder_path_q, usgs_df_paths_q, gage_idx)
            usgs_col = "Q"

        plot_cldnet_vals = cldnet_sim_data.values
        plot_synxflow_vals = synxflow_sim_data.values
        if metric_name == "wse" and shift_mean and not usgs_df.empty:
            usgs_mean = usgs_df[usgs_col].mean()
            plot_cldnet_vals = cldnet_sim_data.values - cldnet_sim_data.mean().values + usgs_mean
            plot_synxflow_vals = synxflow_sim_data.values - synxflow_sim_data.mean().values + usgs_mean
        ax.plot(
            cldnet_sim_data["time"],
            plot_cldnet_vals,
            marker="s",
            markersize=3,
            c="b",
            alpha=0.7,
            linewidth=2,
            label="CLDNet"
        )
        ax.plot(
            synxflow_sim_data["time"],
            plot_synxflow_vals,
            marker="o",
            markersize=3,
            c="r",
            alpha=0.7,
            linewidth=2,
            label="SynxFlow"
        )
        ax.plot(usgs_df["datetime"], usgs_df[usgs_col], ls="-", c="k", linewidth=2, label="USGS Hourly")

        ax.set_title(f"Gage {gage_idx}", fontweight='bold')
        ax.set_ylabel(y_label)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(frameon=False)
        ax.tick_params(axis='x', rotation=30)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    plt.tight_layout()
    plt.savefig(os.path.join(figdir, filename), bbox_inches='tight')
    print(f"Saved: {filename}")

# plot_gauge_comparison("wse", "WSE [m]", "wse_main_river_gages.png")
# plot_gauge_comparison("wse", "WSE [m]", "wse_mean_shifted_main_river_gages.png", shift_mean=True)
# plot_gauge_comparison("Q", "Q [m³/s]", "q_main_river_gages.png")


In [ ]:

# Use the requested hourly comparison window for everything
target_time = comparison_start_datetime
lw = 4
text_rotation = 20
for gage_idx in main_river_gauges_to_plot:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5), dpi=200)

    width = 90 if gage_idx in width_90_gauges else 60
    cldnet_ds = cldnet_datasets[width]
    synxflow_ds = synxflow_datasets[width]

    cldnet_time_values = pd.to_datetime(cldnet_ds.time.values)
    synxflow_time_values = pd.to_datetime(synxflow_ds.time.values)
    cldnet_start_index = np.searchsorted(cldnet_time_values, target_time)
    synxflow_start_index = np.searchsorted(synxflow_time_values, target_time)

    # --- LEFT PLOT: WSE ---
    cldnet_wse = cldnet_ds["wse"].isel(gauge=gage_idx, frame=slice(cldnet_start_index, None))
    synxflow_wse = synxflow_ds["wse"].isel(gauge=gage_idx, frame=slice(synxflow_start_index, None))
    usgs_wse_df = get_usgs_h_wse(usgs_folder_path_h, usgs_df_paths_h, gage_idx)

    plot_cldnet_wse_vals = cldnet_wse.values
    plot_synxflow_wse_vals = synxflow_wse.values
    if not usgs_wse_df.empty:
        usgs_wse_mean = usgs_wse_df["wse"].mean()
        plot_cldnet_wse_vals = cldnet_wse.values - cldnet_wse.mean().values + usgs_wse_mean
        plot_synxflow_wse_vals = synxflow_wse.values - synxflow_wse.mean().values + usgs_wse_mean

    ax1.plot(synxflow_wse["time"], plot_synxflow_wse_vals, marker="o", ms=8, c="r", lw=lw, alpha=0.7, label="SynxFlow")
    ax1.plot(usgs_wse_df["datetime"], usgs_wse_df["wse"], ls="-", c="k", lw=lw, label="USGS Hourly")
    ax1.set_title("SynxFlow vs USGS [WSE m]")
    ax1.legend(frameon=False)
    ax1.grid(True, ls='--', alpha=0.5)
    ax1.tick_params(axis='x', rotation=text_rotation, labelsize=12)
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    # --- RIGHT PLOT: Discharge ---
    cldnet_q = cldnet_ds["Q"].isel(gauge=gage_idx, frame=slice(cldnet_start_index, None))
    synxflow_q = synxflow_ds["Q"].isel(gauge=gage_idx, frame=slice(synxflow_start_index, None))
    usgs_q_df = get_usgs_q(usgs_folder_path_q, usgs_df_paths_q, gage_idx)

    # ax2.plot(cldnet_q["time"], cldnet_q.values, marker="s", ms=8, c="b", lw=lw, alpha=0.7, label="CLDNet")
    # ax2.plot(synxflow_q["time"], synxflow_q.values, marker="o", ms=8, c="r", lw=lw, alpha=0.7, label="SynxFlow")
    # ax2.plot(usgs_q_df["datetime"], usgs_q_df["Q"], ls="-", c="k", lw=lw, label="USGS Hourly")
    # ax2.set_title("Q [m³/s]")
    ax2.plot(cldnet_wse["time"], plot_cldnet_wse_vals, marker="s", ms=8, c="b", lw=lw, alpha=0.7, label="CLDNet")
    ax2.plot(synxflow_wse["time"], plot_synxflow_wse_vals, marker="o", ms=8, c="r", lw=lw, alpha=0.7, label="SynxFlow")
    ax2.set_title("CLDNet vs SynxFlow [WSE m]")
    ax2.legend(frameon=False)
    ax2.grid(True, ls='--', alpha=0.5)
    ax2.tick_params(axis='x', rotation=text_rotation, labelsize=12)
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    site_no = usgs_wse_df['site_no'].iloc[0] if not usgs_wse_df.empty else usgs_df_paths_h[gage_idx].split("_")[1]
    plt.suptitle(rf'USGS Gauge ID: {site_no}')

    plt.tight_layout()
    plt.savefig(os.path.join(figdir, f"gage_{gage_idx}_comparison.png"), bbox_inches='tight')
    plt.show()
    plt.close(fig)

    # Keep the WSE-only figure at roughly one panel wide so it matches the original layout.
    fig_wse, ax_wse = plt.subplots(figsize=(6, 3.5), dpi=200)
    ax_wse.plot(cldnet_wse["time"], plot_cldnet_wse_vals, marker="s", ms=8, c="b", lw=lw, alpha=0.7, label="CLDNet")
    ax_wse.plot(synxflow_wse["time"], plot_synxflow_wse_vals, marker="o", ms=8, c="r", lw=lw, alpha=0.7, label="SynxFlow")
    ax_wse.plot(usgs_wse_df["datetime"], usgs_wse_df["wse"], ls="-", c="k", lw=lw, label="USGS Hourly")
    ax_wse.set_title("WSE Comparison")
    ax_wse.set_ylabel("WSE [m]")
    ax_wse.legend(frameon=False)
    ax_wse.grid(True, ls='--', alpha=0.5)
    ax_wse.tick_params(axis='x', rotation=text_rotation, labelsize=12)
    ax_wse.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    fig_wse.suptitle(rf'USGS Gauge ID: {site_no}')
    fig_wse.tight_layout(rect=[0, 0, 1, 0.94])
    fig_wse.savefig(os.path.join(figdir, f"gage_{gage_idx}_wse_comparison.png"), bbox_inches='tight')
    plt.close(fig_wse)
    print(f"Saved WSE-only comparison: gage_{gage_idx}_wse_comparison.png")
    print(f"Success: Gage {gage_idx}")

for ds in cldnet_datasets.values():
    ds.close()
for ds in synxflow_datasets.values():
    ds.close()


In [ ]:
# --- 1. Generate the Flood Polygons ---
# We use the reprojected ds_4326 from your previous step
frame_numbers = [950400]
frame_numbers = [950400, 1166400]
usgs_gauges_df = pd.read_csv(parent_dir+'/gage_index_mapping_with_lat_lon.csv')
for frame_number  in frame_numbers:
    # frame_path = synxflow_dir + rf'/frame_{frame_number}.nc'
    # ds = xr.open_dataset(frame_path)
    # 1. Load and assign initial CRS
    frame_path = f"{synxflow_dir}/frame_{frame_number}.nc"
    ds = xr.open_dataset(frame_path)
    ds = ds.rio.write_crs("EPSG:5070") # Set original projection (HiPIMS default)

    # 2. Reproject to Lon/Lat (WGS84)
    ds_4326 = ds.rio.reproject("EPSG:4326")
    
    depth_threshold = 0.15
    mask = (ds_4326["h"] > depth_threshold).astype(np.uint8)

    marker_size = 80
    font_size = 12

    # Convert raster to shapes
    shapes = features.shapes(mask.values, transform=ds_4326.rio.transform())
    geoms = [Polygon(s[0]["coordinates"][0]) for s in shapes if s[1] == 1]
    gdf = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")

    # --- 2. Setup Plot with safe DPI ---
    fig, ax = plt.subplots(figsize=(8, 12), dpi=300) 

    # --- 3. Plot the Flood Layer ---
    if not gdf.empty:
        gdf.plot(ax=ax, color="cyan", alpha=0.5, edgecolor="blue", linewidth=0.3, zorder=2)

    # --- 4. Plot USGS Gauges ---
    for _, usgs_gauge in usgs_gauges_df.iterrows():
        # Double check: ensure gauge is in degrees (-180 to 180)
        if int(usgs_gauge['i']) in main_river_gauges_to_plot:
            lon, lat = usgs_gauge['lon'], usgs_gauge['lat']
            # Safety check for projection outliers
            if -180 < lon < 180 and -90 < lat < 90:
                ax.scatter(lon, lat, color='orange', marker='^', s=marker_size, 
                        edgecolor='black', zorder=5)
                # random_shift =  random.uniform(-0.02,0.02)
                ax.text(lon, lat , str(int(usgs_gauge['staid'])), 
                        fontsize=font_size, fontweight='bold', zorder=6)

    # --- 5. Plot Photo Locations ---
    for name, (lat, lon) in real_photo_locations.items():
        ax.scatter(lon, lat, color='red', marker='s', s=marker_size, 
                edgecolor='black', zorder=7)

    # --- 6. CRITICAL: Set Axis Limits BEFORE adding basemap ---
    # This prevents the "Image too large" error by clipping the render area
    minx, miny, maxx, maxy = gdf.total_bounds
    ax.set_xlim(minx - 0.02, maxx + 0.02)
    ax.set_ylim(miny - 0.02, maxy + 0.02)

    # --- 7. Add Basemap ---
    cx.add_basemap(
        ax, 
        crs="EPSG:4326", 
        source=cx.providers.CartoDB.Positron,
        attribution="",
        zoom='auto' # Contextily will pick the best resolution for these limits
    )

    # --- 8. Final Polish ---
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, f"polygon_map.png"), bbox_inches='tight')

    plt.show()

    ############## Lets savve the kml file 
    # --- Inside your frame loop ---
    if not gdf.empty:
        # 1. Project to WGS84 (Google Earth standard)
        gdf_kml = gdf.to_crs(epsg=4326).copy()
        
        # 2. Initialize the KML object
        kml = simplekml.Kml()
        
        # 3. Handle MultiPolygons by "exploding" them into individual Polygons
        # SimpleKML works best when adding one polygon at a time
        gdf_exploded = gdf_kml.explode(index_parts=False)
        
        for i, row in gdf_exploded.iterrows():
            # Extract coordinates for the outer boundary
            # Format: [(lon, lat), (lon, lat), ...]
            coords = list(row.geometry.exterior.coords)
            
            # Create the polygon in KML
            pol = kml.newpolygon(name=f"Flood_Area_{i}")
            pol.outerboundaryis = coords
            
            # 4. Apply Cyan Styling
            # Color format is AABBGGRR (Alpha-Blue-Green-Red) in hex
            # 'b2' is ~70% opacity, 'ffff' is max Blue+Green (Cyan), '00' is no Red
            pol.style.polystyle.color = 'b2ffff00'  # Transparent Cyan fill
            # pol.style.linestyle.color = 'ff0000ff'   # Solid Blue border
            pol.style.linestyle.width = 1            # Thin border line
            
            # Ensure it sticks to the ground terrain
            # pol.altitudemode = simplekml.AltitudeMode.clamptoground

        # 5. Save the KML file
        kml_filename = f"flood_extent_frame_{frame_number}_threshold_{depth_threshold}.kml"
        kml_save_path = os.path.join(kmldir, kml_filename)
        kml.save(kml_save_path)
        
        print(f"🌎 Styled KML (Cyan) saved to: {kml_save_path}")
